# Clinical NLP Pipeline — Google Colab

**Predicting 30-Day Hospital Readmission from Discharge Notes**

> **Runtime:** This pipeline is CPU-bound. A GPU runtime will NOT speed it up. Use `Runtime → Change runtime type → CPU`.

## 0. Environment Setup

In [ ]:
import os, sys
REPO_URL = "https://github.com/IamPrashantBhattarai/Clinical-NLP.git"
REPO_DIR = "/content/Clinical-NLP"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print("CWD:", os.getcwd())

In [ ]:
# Install dependencies — skip heavy unused packages (bertopic, hdbscan, umap-learn)
!pip install -q pandas numpy scipy PyYAML nltk spacy gensim scikit-learn \
    xgboost lightgbm imbalanced-learn fairlearn \
    matplotlib seaborn plotly wordcloud tqdm \
    ipywidgets textblob faker pyLDAvis transformers tokenizers

In [ ]:
!python -m spacy download en_core_web_sm -q

import nltk
for res in ["stopwords", "wordnet", "punkt", "punkt_tab", "averaged_perceptron_tagger"]:
    nltk.download(res, quiet=True)
print("Models ready.")

In [ ]:
import sys, logging, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("colab_pipeline")
logger.info("Environment ready.")

## 0b. (Optional) Mount Google Drive

In [ ]:
SAVE_TO_DRIVE = True  # set to False to skip

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT = Path("/content/drive/MyDrive/ClinicalNLP_results")
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print("Results will be copied to:", DRIVE_OUT)
else:
    DRIVE_OUT = None

## 1. Data Generation & Loading

In [ ]:
import yaml
from src.generate_synthetic_data import run as generate_data
from src.data_loader import load_all, get_data_summary

CFG_PATH = Path("config/config.yaml")
config = yaml.safe_load(CFG_PATH.read_text()) if CFG_PATH.exists() else {}

generate_data(output_dir="data", n_patients=200)
df = load_all(config, use_synthetic=True)
summary = get_data_summary(df)

print(f"\nDataset shape: {df.shape}")
df.head(3)

## 2. Preprocessing

In [ ]:
%%time
from src.preprocess import build_preprocessing_pipeline

processed = build_preprocessing_pipeline(df, config=config, use_scispacy=False, use_phrases=False)

print(f"Processed shape: {processed.shape}")
print(f"Avg tokens per note: {processed['num_tokens'].mean():.1f}")
print(f"\nSample tokens: {processed['tokens'].iloc[0][:15]}")

## 3. Topic Modeling (LDA)

In [ ]:
from src.topic_model import run_lda_pipeline

eligible_mask = processed["readmission_30day"] >= 0
eligible_df = processed[eligible_mask].reset_index(drop=True)
tokenized_docs = eligible_df["tokens"].apply(lambda t: t if isinstance(t, list) else []).tolist()

print(f"Eligible documents for topic modeling: {len(tokenized_docs)}")

lda_results = run_lda_pipeline(tokenized_docs, config=config)

print(f"\nBest number of topics: {lda_results['best_num_topics']}")
print(f"Coherence scores: {lda_results['coherence_scores']}")
lda_results["topics_df"]

## 4. Feature Engineering

In [ ]:
from src.feature_engineer import build_feature_sets

feature_sets = build_feature_sets(processed, lda_results=lda_results, config=config)

print("Feature sets built:")
for key in ["tfidf", "topic_lda", "structured", "text_stats", "combined"]:
    entry = feature_sets.get(key)
    if entry and entry.get("X") is not None:
        print(f"  {key:20s} — shape: {entry['X'].shape}")

print(f"\nLabels: {len(feature_sets['label'])} samples")
print(f"  Readmitted: {feature_sets['label'].sum()} ({feature_sets['label'].mean()*100:.1f}%)")

## 5. Prediction Pipeline

In [ ]:
from src.predict import run_prediction_pipeline

prediction_results = run_prediction_pipeline(feature_sets, config=config)

# Results table
display_cols = ["model", "feature_type", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
results_df = prediction_results["results_df"]
results_df[display_cols].sort_values("roc_auc", ascending=False)

In [ ]:
# Best model summary
best = prediction_results["best"]
print(f"Best model: {best['model']} + {best['feature_type']}")
print(f"  ROC-AUC: {best['roc_auc']:.4f}")
print(f"  F1:      {best['f1']:.4f}")
print(f"  PR-AUC:  {best['pr_auc']:.4f}")

# Cross-validation results
if prediction_results["cv_results"]:
    print("\nCross-validation results:")
    for cv in prediction_results["cv_results"]:
        print(f"  {cv['model']} + {cv['feature_type']} — "
              f"F1: {cv['f1_mean']:.3f}\u00b1{cv['f1_std']:.3f} | "
              f"ROC-AUC: {cv['roc_auc_mean']:.3f}\u00b1{cv['roc_auc_std']:.3f}")

## 6. Fairness Audit

In [ ]:
from src.fairness import run_fairness_audit
from sklearn.model_selection import train_test_split

best_model_name = best["model"]
best_feat_type = best["feature_type"]
best_model = prediction_results["models"].get((best_model_name, best_feat_type))
best_splits = prediction_results["splits"].get(best_feat_type)

fairness_results = None
if best_model and best_splits:
    y_test = best_splits["y_test"]
    X_test = best_splits["X_test"]
    y_prob = best_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    # Re-derive test indices to align protected attributes with test set
    labels = feature_sets["label"]
    test_size = config.get("prediction", {}).get("test_size", 0.2)
    _, test_idx, _, _ = train_test_split(
        np.arange(len(labels)), labels,
        test_size=test_size, stratify=labels, random_state=42,
    )

    eligible_df = feature_sets["eligible_df"]
    protected_cols = config.get("fairness", {}).get("protected_attributes", ["gender", "insurance", "age_group"])
    protected_df = eligible_df.loc[test_idx, protected_cols].reset_index(drop=True)

    fairness_results = run_fairness_audit(
        y_true=y_test,
        y_pred=y_pred,
        y_prob=y_prob,
        protected_df=protected_df,
        config=config,
    )
else:
    print("Best model not available for fairness audit.")

In [ ]:
if fairness_results:
    fairness_results["summary_df"]

## 7. Visualization

In [ ]:
from src.visualize import generate_all_figures

lda_results["readmission_labels"] = feature_sets["label"]

saved_figures = generate_all_figures(
    merged_df=df,
    prediction_results=prediction_results,
    lda_results=lda_results,
    fairness_results=fairness_results,
    config=config,
)

print(f"Generated {len(saved_figures)} figures:")
for fig_path in saved_figures:
    print(f"  {fig_path}")

In [ ]:
from IPython.display import Image, display

for name in ["demographics", "roc_curves", "model_comparison", "fairness_disparity"]:
    fig_path = Path(f"results/figures/{name}.png")
    if fig_path.exists():
        print(f"\n{'='*60}")
        print(f"  {name.replace('_', ' ').title()}")
        print(f"{'='*60}")
        display(Image(filename=str(fig_path), width=800))

## 8. Save Results to Drive

In [ ]:
import shutil, joblib

if DRIVE_OUT is not None:
    # Figures
    fig_src = Path("results/figures")
    if fig_src.exists():
        shutil.copytree(fig_src, DRIVE_OUT / "figures", dirs_exist_ok=True)

    # Save best model
    models_out = DRIVE_OUT / "models"
    models_out.mkdir(exist_ok=True)
    best_model_obj = prediction_results["models"].get((best["model"], best["feature_type"]))
    if best_model_obj is not None:
        model_file = models_out / f"{best['model']}_{best['feature_type']}.joblib"
        joblib.dump(best_model_obj, model_file)
        print(f"Saved best model -> {model_file}")

    # Save processed notes
    try:
        processed.to_parquet(DRIVE_OUT / "processed_notes.parquet")
        print("Saved processed notes parquet")
    except Exception as e:
        processed.to_csv(DRIVE_OUT / "processed_notes.csv", index=False)
        print("Parquet failed, saved CSV instead:", e)

    print(f"\nAll results in: {DRIVE_OUT}")
else:
    print("SAVE_TO_DRIVE is False — results remain in /content only.")

---

## Summary

| Stage | Output |
|-------|--------|
| Data  | Synthetic MIMIC-IV (200 patients, ~300+ notes) |
| Preprocessing | PHI removal, spaCy tokenization (batched nlp.pipe), clinical stopwords |
| Topic model | LDA with coherence-based topic selection |
| Features | TF-IDF, topic distributions, structured, text stats, combined |
| Models | Logistic Regression, Random Forest, XGBoost, LightGBM |
| Fairness | Demographic parity, equalized odds, FPR/FNR disparity |
| Figures | `results/figures/*.png` (copied to Drive) |